# Kaleido — Pipeline 3 (v5): Curriculum Assembly

**v5 changes (2026-03-23):**
- 14 practices (was 12) — adds `POV_INTRO`, `L3M_POV`, `L2M_POV`; retires old `L3M`
- `DIRECTION_LOOKUP` loads `argument`, `logic`, `blog_url` (was `argument` only)
- `L4M` reduced to 2 questions (structure question removed); `context` field added to all MCQ
- `L2M` rhetoric-only, body sentences, cap 6; Template B (PoV) moved to `L2M_POV`
- `L1M` body sentences only, first noun-phrase per sentence
- `L1S` capped at 10; `L1F` body only, cap 10; `L2F` body only, cap 5
- Aligns with Contract v3 + practice_content_spec v2 + practice_redesign_spec v3

---
## Cell 1 — Install dependencies

In [1]:
!pip install supabase spacy --quiet
!python -m spacy download en_core_web_sm --quiet
print('✅ Dependencies ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 716.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
✅ Dependencies ready


---
## Cell 2 — Your Supabase credentials

In [ ]:
SUPABASE_URL   = "https://yxxmjptlkeewcabczlgt.supabase.co"
SUPABASE_KEY   = "sb_publishable_VcS5cSLESM977M6IQpcscA_TsqUx5Es"

---
## Cell 3 — Connect to Supabase + load spaCy

In [3]:
from supabase import create_client
import spacy

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
nlp = spacy.load('en_core_web_sm')

print('✅ Supabase connected')
print('✅ spaCy loaded')

✅ Supabase connected
✅ spaCy loaded


---
## Cell 4 — Load P2 enriched snapshot from Google Drive

Downloads `p2_enriched_snapshot.json` — the file P2 uploaded in its final cell.

In [4]:
import json
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

auth.authenticate_user()
service = build('drive', 'v3')

FOLDER_ID = '1eOMwvWuJw_BsOaQXcgTKVLx2ykK77UXE'
FILE_NAME = 'p2_enriched_snapshot.json'

files = service.files().list(
    q=f"name='{FILE_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id)'
).execute().get('files', [])

if not files:
    raise FileNotFoundError(f'❌ {FILE_NAME} not found in Drive — run P2 first')

request = service.files().get_media(fileId=files[0]['id'])
with open(FILE_NAME, 'wb') as f:
    downloader = MediaIoBaseDownload(f, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()

print(f'✅ Downloaded {FILE_NAME}')

with open(FILE_NAME, 'r', encoding='utf-8') as f:
    snapshot = json.load(f)

# Sanity checks
assert 'batch_id' in snapshot, '❌ Missing batch_id'
assert 'tier_50'  in snapshot, '❌ Missing tier_50'
assert 'tier_80'  in snapshot, '❌ Missing tier_80'

BATCH_ID = snapshot['batch_id']

print(f'batch_id      : {BATCH_ID}')
print(f'generated_at  : {snapshot.get("generated_at", "?")}')
print(f'tier_50 steps : {len(snapshot["tier_50"]["sequence"])}')
print(f'tier_80 steps : {len(snapshot["tier_80"]["sequence"])}')
print(f'tier_50 pool  : {len(snapshot["tier_50"]["pool"])} essays')
print(f'tier_80 pool  : {len(snapshot["tier_80"]["pool"])} essays')
print('\n✅ Snapshot loaded — ready for assembly')

✅ Downloaded p2_enriched_snapshot.json
batch_id      : 00b60e61-22d9-4513-8d08-a8f7c09e6c79
generated_at  : 2026-03-12T03:00:58.303884Z
tier_50 steps : 6
tier_80 steps : 14
tier_50 pool  : 53 essays
tier_80 pool  : 75 essays

✅ Snapshot loaded — ready for assembly


---
## Cell 5 — Define lookup tables

Two hardcoded dictionaries used by assembler functions.
(DIRECTION_LOOKUP is now in Supabase `direction_ref` table — fetched here.)

In [ ]:
# ── STRUCTURE_LOOKUP (7 entries) ───────────────────────────────────────────
STRUCTURE_LOOKUP = {
    "structure_1": "4-Paragraph Discuss Both Views",
    "structure_2": "5-Paragraph Discuss Both Views + Opinion",
    "structure_3": "4-Paragraph Opinion-Led",
    "structure_4": "5-Paragraph Opinion + Counterargument",
    "structure_5": "4-Paragraph Advantages–Disadvantages",
    "structure_6": "4-Paragraph Problem–Solution",
    "structure_7": "5-Paragraph Causes–Effects–Solutions",
}
print(f'✅ STRUCTURE_LOOKUP — {len(STRUCTURE_LOOKUP)} entries')

# ── RHETORIC_LABEL_LOOKUP (33 entries) ────────────────────────────────────
RHETORIC_LABEL_LOOKUP = {
    "paraphrase_question":       "Introduction — restates the question in new words",
    "outline_statement":         "Introduction — previews the essay structure",
    "clear_thesis_statement":    "Introduction — states the writer's direct position",
    "topic_sentence":            "Body — introduces the paragraph's main claim",
    "explanation":               "Body — unpacks the causal mechanism",
    "example_1":                 "Body — first concrete evidence",
    "example_2":                 "Body — second concrete evidence",
    "mini_conclusion":           "Body — summarises the paragraph's logic",
    "link_to_thesis":            "Body — connects paragraph back to main argument",
    "opinion_topic_sentence":    "Opinion — opens with the writer's stance",
    "reason_1":                  "Opinion — first reason supporting the opinion",
    "reason_2":                  "Opinion — second reason supporting the opinion",
    "synthesis":                 "Opinion — combines ideas into a nuanced conclusion",
    "acknowledge_opposing_view": "Counterargument — introduces the opposing position",
    "concession":                "Counterargument — admits a valid point in opposition",
    "rebuttal":                  "Counterargument — refutes the opposing argument",
    "return_to_thesis":          "Counterargument — reasserts the writer's position",
    "problem_1":                 "Body — first problem identified",
    "problem_2":                 "Body — second problem identified",
    "solution_1":                "Body — first solution proposed",
    "solution_2":                "Body — second solution proposed",
    "cause_1":                   "Body — first cause identified",
    "cause_2":                   "Body — second cause identified",
    "effect_1":                  "Body — first effect described",
    "effect_2":                  "Body — second effect described",
    "advantage_1":               "Body — first advantage stated",
    "advantage_2":               "Body — second advantage stated",
    "disadvantage_1":            "Body — first disadvantage stated",
    "disadvantage_2":            "Body — second disadvantage stated",
    "opinion_statement":         "Conclusion — states the writer's final opinion",
    "restate_thesis":            "Conclusion — restates the main argument",
    "summary_statement":         "Conclusion — summarises the key points",
    "which_outweighs_opinion":   "Conclusion — declares which side is stronger",
}
print(f'✅ RHETORIC_LABEL_LOOKUP — {len(RHETORIC_LABEL_LOOKUP)} entries')

# ── DIRECTION_LOOKUP (from Supabase direction_ref table) ──────────────────
# v5: now loads argument + logic + blog_url (was argument-only)
dref_rows = supabase.table('direction_ref').select('*').execute().data
DIRECTION_LOOKUP = {
    r['direction_id']: {
        'argument': r['argument'],
        'logic':    r['logic'],
        'blog_url': r.get('blog_url'),
    }
    for r in dref_rows
}
print(f'✅ DIRECTION_LOOKUP — {len(DIRECTION_LOOKUP)} entries loaded from Supabase (argument + logic + blog_url)')

---
## Cell 6 — Utility functions

Five reusable helpers used by the assembler functions in Cell 7.

In [ ]:
import re
import random

# ── Case-insensitive phrase finder (shared by L1M, L2S, L1F) ───────────────
def find_phrase_in_text(phrase, text):
    """
    Locates a phrase in text using case-insensitive matching.
    Returns the phrase as it actually appears in text (preserving original casing),
    or None if the phrase is not found.
    """
    match = re.search(re.escape(phrase), text, re.IGNORECASE)
    return match.group() if match else None


# ── POS tagger ──────────────────────────────────────────────────────────────
_POS_MAP = {
    'NOUN': 'NOUN', 'PROPN': 'NOUN',
    'VERB': 'VERB', 'AUX':   'VERB',
    'ADJ':  'ADJ',  'ADV':   'ADV',
}

def pos_tag(phrase):
    """Returns POS of a phrase: NOUN, VERB, ADJ, ADV, or PHRASE."""
    doc = nlp(phrase)
    root_tokens = [t for t in doc if t.dep_ == 'ROOT']
    if not root_tokens:
        return 'PHRASE'
    return _POS_MAP.get(root_tokens[0].pos_, 'PHRASE')


# ── Direction helpers (v5: DIRECTION_LOOKUP is now {tag: {argument, logic, blog_url}}) ──
def dir_argument(tag):
    """Returns the argument text for a direction_tag, or the tag itself as fallback."""
    entry = DIRECTION_LOOKUP.get(tag)
    return entry['argument'] if entry else tag

def dir_logic(tag):
    """Returns the logic text for a direction_tag."""
    entry = DIRECTION_LOOKUP.get(tag)
    return entry['logic'] if entry else ''

def dir_blog_url(tag):
    """Returns the blog_url for a direction_tag."""
    entry = DIRECTION_LOOKUP.get(tag)
    return entry.get('blog_url') if entry else None


# ── Dominant direction of a paragraph ────────────────────────────────────────
def dominant_direction(sentences):
    """
    Returns the dominant direction_tag for a list of sentences (one paragraph).
    Most frequent direction_tag. Tie → earliest sentence's tag (lowest order).
    """
    from collections import Counter
    tags = [s.get('direction_tag') for s in sentences if s.get('direction_tag')]
    if not tags:
        return None
    counts = Counter(tags)
    max_count = max(counts.values())
    tied = [t for t, c in counts.items() if c == max_count]
    if len(tied) == 1:
        return tied[0]
    # Tie-break: earliest sentence's tag
    for s in sorted(sentences, key=lambda s: s['order']):
        if s.get('direction_tag') in tied:
            return s['direction_tag']
    return tied[0]


# ── Sentence chunker for L2S (v5-fix: spaCy phrase-level chips) ─────────────
def chunk_sentence(canonical_text, lexical_items):
    """
    Splits a sentence into scramble chips for L2S.
    Lexical phrases → single chip (is_lexical: True).
    Remaining text → clause/phrase-level chips via spaCy (is_lexical: False).
    Target non-lexical chip size: 2–5 words.
    """
    phrases = [
        li['phrase'] if isinstance(li, dict) else li
        for li in lexical_items
    ]
    sorted_phrases = sorted(phrases, key=len, reverse=True)

    # Step 1: tokenise the sentence into spans, protecting lexical phrases
    spans = []  # list of {'text': str, 'is_lexical': bool}
    remaining = canonical_text

    while remaining:
        matched = False
        for phrase in sorted_phrases:
            actual = find_phrase_in_text(phrase, remaining[:len(phrase) + 5])
            if actual and remaining[:len(actual)].lower() == actual.lower():
                spans.append({'text': remaining[:len(actual)], 'is_lexical': True})
                remaining = remaining[len(actual):].lstrip(' ')
                matched = True
                break
        if not matched:
            # Find next lexical phrase boundary or end of string
            next_boundary = len(remaining)
            for phrase in sorted_phrases:
                idx = remaining.lower().find(phrase.lower())
                if idx > 0:
                    next_boundary = min(next_boundary, idx)
            non_lex_text = remaining[:next_boundary].strip()
            if non_lex_text:
                spans.append({'text': non_lex_text, 'is_lexical': False})
            remaining = remaining[next_boundary:].lstrip(' ')

    # Step 2: for each non-lexical span, use spaCy to split into phrase chips
    chunks = []
    for span in spans:
        if span['is_lexical']:
            chunks.append(span)
        else:
            doc = nlp(span['text'])
            phrase_chips = _spacy_phrase_chunks(doc)
            for chip in phrase_chips:
                chunks.append({'text': chip, 'is_lexical': False})

    # Fix 3: merge standalone punctuation chips into an adjacent chunk.
    # Trailing punct (e.g. '.') -> merge into preceding chunk.
    # Leading punct (e.g. ',' starting a non-lexical span) -> merge into following chunk.
    _STANDALONE_PUNCT = {'.', ',', ';', ':', '!', '?'}
    merged = []
    for chunk in chunks:
        if chunk['text'] in _STANDALONE_PUNCT and merged:
            # Trailing: attach to preceding chunk
            merged[-1] = {
                'text': merged[-1]['text'] + chunk['text'],
                'is_lexical': merged[-1]['is_lexical'],
            }
        else:
            merged.append(chunk)
    # Forward pass: leading punct with nothing preceding -> attach to next chunk
    i = 0
    final = []
    while i < len(merged):
        if merged[i]['text'] in _STANDALONE_PUNCT and i + 1 < len(merged):
            final.append({
                'text': merged[i]['text'] + merged[i + 1]['text'],
                'is_lexical': merged[i + 1]['is_lexical'],
            })
            i += 2
        else:
            final.append(merged[i])
            i += 1
    chunks = final

    return chunks


def _spacy_phrase_chunks(doc):
    """
    Groups spaCy tokens into phrase-level chips using dependency structure.
    Groups: noun chunks, verb+aux groups, prep phrases, punctuation standalone.
    Returns list of strings.
    """
    used = set()
    chips = []

    # Pass 1: noun chunks
    for nc in doc.noun_chunks:
        chip = nc.text.strip()
        if chip:
            chips.append((nc.start, chip))
            for i in range(nc.start, nc.end):
                used.add(i)

    # Pass 2: verb groups (aux + verb head)
    for token in doc:
        if token.i in used:
            continue
        if token.pos_ in ('VERB', 'AUX') and token.dep_ in ('ROOT', 'relcl', 'advcl', 'xcomp'):
            group = [token]
            for child in token.children:
                if child.pos_ == 'AUX' and child.i not in used:
                    group.append(child)
            group_sorted = sorted(group, key=lambda t: t.i)
            # Fix 2: use contiguous doc span to avoid skipping intermediate words.
            # e.g. 'does little to mitigate' instead of 'does mitigate'.
            start = group_sorted[0].i
            end   = group_sorted[-1].i + 1
            chip  = doc[start:end].text.strip()
            if chip:
                chips.append((start, chip))
            # Mark ALL tokens in the span as used (not just AUX+verb members)
            for t in doc[start:end]:
                used.add(t.i)

    # Pass 3: prepositional phrases (prep + pobj subtree)
    # Fix 4: only include subtree tokens not already used — prevents overlap
    # with Pass 1 noun chunks (e.g. 'to improved academic outcomes' duplicating
    # 'improved academic outcomes' already claimed by Pass 1).
    for token in doc:
        if token.i in used:
            continue
        if token.dep_ == 'prep':
            subtree_tokens = sorted(
                [t for t in token.subtree if t.i not in used],
                key=lambda t: t.i,
            )
            chip = ' '.join(t.text for t in subtree_tokens).strip()
            if chip:
                chips.append((subtree_tokens[0].i, chip))
            for t in subtree_tokens:
                used.add(t.i)

    # Pass 4: remaining ungrouped tokens (fallback — single words or punct)
    for token in doc:
        if token.i not in used:
            chips.append((token.i, token.text))

    # Sort by original position and return text only
    chips.sort(key=lambda x: x[0])
    return [c[1] for c in chips if c[1].strip()]


# ── Distractor pool builder ─────────────────────────────────────────────────
def build_distractor_pools(step_n, tier_data):
    """
    Builds 4 priority buckets of candidate sentences for step N.
    Pool 1: challenge (always empty post-redesign)
    Pool 2: next-step essays
    Pool 3: previous-step essays
    Pool 4: all remaining tier pool essays
    """
    sequence = tier_data['sequence']
    pool     = tier_data['pool']
    step     = sequence[step_n]

    used_essay_ids = set()

    # Pool 1: challenge (always empty)
    pool1 = []
    if step.get('challenge'):
        pool1 = step['challenge']['sentences']
        used_essay_ids.add(step['challenge']['essay_id'])
    if step.get('model'):
        used_essay_ids.add(step['model']['essay_id'])

    # Pool 2: next-step essays
    pool2 = []
    if step_n + 1 < len(sequence):
        next_step = sequence[step_n + 1]
        if next_step.get('model'):
            pool2 += next_step['model']['sentences']
            used_essay_ids.add(next_step['model']['essay_id'])

    # Pool 3: previous-step essays
    pool3 = []
    if step_n - 1 >= 0:
        prev_step = sequence[step_n - 1]
        if prev_step.get('model'):
            pool3 += prev_step['model']['sentences']
            used_essay_ids.add(prev_step['model']['essay_id'])

    # Pool 4: all remaining
    pool4 = []
    for essay in pool:
        if essay['essay_id'] not in used_essay_ids:
            pool4 += essay['sentences']

    return [pool1, pool2, pool3, pool4]


# ── Distractor selector ─────────────────────────────────────────────────────
def get_distractors(tag, pools, n=4, exclude_ids=None):
    """Picks n distractor sentences matching rhetoric_tag, with Priority 5 fallback."""
    exclude_ids = exclude_ids or set()
    candidates  = []
    seen_ids    = set(exclude_ids)

    for pool in pools:
        for sentence in pool:
            sid = sentence['sentence_id']
            if sid not in seen_ids and sentence.get('rhetoric_tag') == tag:
                candidates.append(sentence)
                seen_ids.add(sid)
        if len(candidates) >= n:
            return candidates[:n]

    # Priority 5: relax tag constraint
    if len(candidates) < n:
        for sentence in pools[3]:
            sid = sentence['sentence_id']
            if sid not in seen_ids:
                candidates.append(sentence)
                seen_ids.add(sid)
            if len(candidates) >= n:
                break

    return candidates[:n]


# ── MCQ option shuffler ─────────────────────────────────────────────────────
def shuffle_options(correct_text, distractors):
    """Packages correct + distractors into shuffled 5-option MCQ."""
    distractor_texts = [
        d['canonical_text'] if isinstance(d, dict) else d
        for d in distractors[:4]
    ]
    options = [correct_text] + distractor_texts
    random.shuffle(options)
    return {
        'options':       options,
        'correct_index': options.index(correct_text),
    }


print('✅ All utility functions ready (v5: direction helpers + dominant_direction + find_phrase_in_text added)')

---
## Cell 7 — All 12 assembler functions

Each function assembles one practice type. The output shapes match the
dev brief §5.4 spec. Key changes from v2:
- MCQ/Scramble/L1F return `versions[]` array
- L2F returns flat questions (no versions)
- P0 and L4W return simple dicts (no versions)

In [ ]:
_PARA_ORDER      = ['introduction', 'body_1', 'body_2', 'body_3', 'conclusion']
_BODY_PARAS      = ['body_1', 'body_2', 'body_3']


def _body_sentences(model):
    """Returns body sentences sorted in essay order (paragraph then sentence order)."""
    return sorted(
        [s for s in model['sentences'] if s['paragraph_type'] in _BODY_PARAS],
        key=lambda s: (_BODY_PARAS.index(s['paragraph_type']), s['order'])
    )


# ═══════════════════════════════════════════════════════════════════════════════
# P0 — Cold Write
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_P0(step):
    return {
        "practice_code": "P0",
        "question": step['model']['question'],
        "prompt":   "Write a complete IELTS Task 2 essay in response to this question:",
    }


# ═══════════════════════════════════════════════════════════════════════════════
# POV_INTRO — PoV Introduction Screen (NEW in v5)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_POV_INTRO(step):
    """
    Collects all unique direction_tags in essay order, looks up argument + logic + blog_url.
    No questions, no versions.
    """
    model = step['model']
    seen_tags = []
    for s in sorted(model['sentences'], key=lambda s: (_PARA_ORDER.index(s['paragraph_type']) if s['paragraph_type'] in _PARA_ORDER else 99, s['order'])):
        dtag = s.get('direction_tag')
        if dtag and dtag not in seen_tags:
            seen_tags.append(dtag)

    directions = []
    for tag in seen_tags:
        directions.append({
            "direction_tag": tag,
            "argument":      dir_argument(tag),
            "logic":         dir_logic(tag),
            "blog_url":      dir_blog_url(tag),
        })

    return {
        "practice_code": "POV_INTRO",
        "directions": directions,
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4W — Essay Write (unit gate)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L4W(step):
    return {"practice_code": "L4W"}


# ═══════════════════════════════════════════════════════════════════════════════
# L1S — Lexical Scramble (v5: capped at 10)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1S(step):
    questions_v1 = []
    questions_v2 = []
    q_num = 0

    for s in step['model']['sentences']:
        for li in s.get('lexical_items', []):
            phrase = li['phrase']
            words  = phrase.split()
            if len(words) < 2:
                continue
            q_num += 1
            if q_num > 10:  # v5: cap at 10
                break
            q_id = f"L1S-{q_num}"

            # V1 and V2 get different shuffles
            chips_v1 = words.copy(); random.shuffle(chips_v1)
            chips_v2 = words.copy(); random.shuffle(chips_v2)

            base = {"id": q_id, "phrase": phrase, "answer": phrase}
            questions_v1.append({**base, "chips": chips_v1})
            questions_v2.append({**base, "chips": chips_v2})
        if q_num > 10:
            break

    return {
        "practice_code": "L1S",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2S — Sentence Scramble (body sentences only)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2S(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for s in model['sentences']:
        if s['paragraph_type'] not in _BODY_PARAS:
            continue
        chunks = chunk_sentence(s['canonical_text'], s.get('lexical_items', []))

        hint = s['syntax_items'][0] if s.get('syntax_items') else RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
        q_id = f"L2S-{s['paragraph_type']}-S{s['order']}"
        base = {"id": q_id, "paragraph": s['paragraph_type'], "sentence_order": s['order'], "hint": hint, "answer": s['canonical_text']}

        c1 = chunks.copy(); random.shuffle(c1)
        c2 = chunks.copy(); random.shuffle(c2)
        questions_v1.append({**base, "chunks": c1})
        questions_v2.append({**base, "chunks": c2})

    return {
        "practice_code": "L2S",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L3S — Paragraph Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L3S(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        hint = ' → '.join(
            RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
            for s in para_sents
        )
        answer_order = [s['sentence_id'] for s in para_sents]
        display = [{'sentence_id': s['sentence_id'], 'text': s['canonical_text']} for s in para_sents]

        d1 = display.copy(); random.shuffle(d1)
        d2 = display.copy(); random.shuffle(d2)

        base = {"id": f"L3S-{para}", "paragraph": para, "hint": hint, "answer_order": answer_order}
        questions_v1.append({**base, "sentences": d1})
        questions_v2.append({**base, "sentences": d2})

    return {
        "practice_code": "L3S",
        "prompt": "Drag the sentences into the correct order for this paragraph.",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4S — Essay Scramble
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1F(step):
    model = step['model']
    questions_v1 = []
    questions_v2 = []

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        for target_s in para_sents:
            if len(questions_v1) >= 10:  # v5: cap at 10
                break
            lex_items = target_s.get('lexical_items', [])
            if not lex_items:
                continue

            # Pre-validate all blanks: dedup similar_phrases, take top 3, skip question if any blank has 0
            validated_blanks = []
            skip_question = False
            for li in lex_items:
                phrase = li['phrase']
                raw_phrases = li.get('similar_phrases', [])

                # Deduplicate preserving similarity order (most similar first)
                seen = set()
                unique_phrases = []
                for p in raw_phrases:
                    if p not in seen:
                        seen.add(p)
                        unique_phrases.append(p)

                top3 = unique_phrases[:3]

                if not top3:
                    skip_question = True
                    break

                validated_blanks.append({
                    'phrase': phrase,
                    'similar_phrases': top3,
                    'pos_hint': pos_tag(phrase),
                })

            if skip_question or not validated_blanks:
                continue  # drop entire question — one or more blanks had no similar phrases

            # Build parts + blanks using validated_blanks (all have top3 guaranteed)
            sorted_phrases = sorted([b['phrase'] for b in validated_blanks], key=len, reverse=True)
            phrase_to_validated = {b['phrase']: b for b in validated_blanks}

            parts = []
            blanks = []
            blank_index = 0
            current_text = ""
            remaining = target_s['canonical_text']

            while remaining:
                matched = False
                for phrase in sorted_phrases:
                    actual = find_phrase_in_text(phrase, remaining[:len(phrase) + 5])
                    if actual and remaining[:len(actual)].lower() == actual.lower():
                        if current_text:
                            parts.append({"type": "text", "text": current_text})
                            current_text = ""
                        b = phrase_to_validated[phrase]
                        parts.append({"type": "blank", "blank_index": blank_index})
                        blanks.append({
                            "index":           blank_index,
                            "answer":          phrase,
                            "pos_hint":        b['pos_hint'],
                            "similar_phrases": b['similar_phrases'],
                        })
                        blank_index += 1
                        remaining = remaining[len(actual):]
                        matched = True
                        break
                if not matched:
                    current_text += remaining[0]
                    remaining = remaining[1:]

            if current_text:
                parts.append({"type": "text", "text": current_text})

            # Context: full paragraph, target has parts
            context = []
            for s in para_sents:
                if s['sentence_id'] == target_s['sentence_id']:
                    context.append({"order": s['order'], "parts": parts, "isTarget": True})
                else:
                    context.append({"order": s['order'], "text": s['canonical_text'], "isTarget": False})

            q_id = f"L1F-{target_s['paragraph_type']}-S{target_s['order']}"

            q = {
                "id": q_id,
                "paragraph": para,
                "sentence_order": target_s['order'],
                "rhetoric_tag": target_s.get('rhetoric_tag', ''),
                "context": context,
                "blanks": blanks,
                "prompt": "Fill in the missing phrases. Use the part of speech hint and similar phrases below each blank to understand what kind of phrase is missing."
            }
            questions_v1.append(q)
            questions_v2.append(q)  # L1F V1 = V2 (same blanks)

        if len(questions_v1) >= 10:
            break

    return {
        "practice_code": "L1F",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2F — Sentence Fill (v5: body sentences only, cap 5, NO versions — no retry)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2F(step):
    model = step['model']
    questions = []

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        for target_s in para_sents:
            if len(questions) >= 5:  # v5: cap at 5
                break
            context = []
            for s in para_sents:
                if s['sentence_id'] == target_s['sentence_id']:
                    context.append({"order": s['order'], "text": "___", "isTarget": True})
                else:
                    context.append({"order": s['order'], "text": s['canonical_text'], "isTarget": False})

            rhetoric_tag  = target_s.get('rhetoric_tag', '')
            rhetoric_hint = RHETORIC_LABEL_LOOKUP.get(rhetoric_tag, rhetoric_tag)
            hint_sentences = target_s.get('hint_sentences', [])

            q_id = f"L2F-{target_s['paragraph_type']}-S{target_s['order']}"

            questions.append({
                "id":             q_id,
                "paragraph":      para,
                "sentence_order": target_s['order'],
                "rhetoric_tag":   rhetoric_tag,
                "rhetoric_hint":  rhetoric_hint,
                "hint_sentences": hint_sentences,
                "context":        context,
                "prompt":         "Fill in the missing sentence. Use the hint sentences and structure label below to understand what kind of sentence belongs here."
            })

        if len(questions) >= 5:
            break

    # L2F has NO versions array — no retry
    return {
        "practice_code": "L2F",
        "questions": questions,
    }


print('✅ Assemblers ready: P0, POV_INTRO, L4W, L1S, L2S, L3S, L4S, L1F, L2F')

---
## Cell 8 — MCQ assembler functions (L1M, L2M, L3M, L4M)

MCQ practices have V1/V2 versions with different distractor priority ordering.

In [ ]:
# ── MCQ helpers ──────────────────────────────────────────────────────────────

def _pools_v2(pools):
    """Swap pool[0] and pool[1] for V2 priority ordering."""
    return [pools[1], pools[0], pools[2], pools[3]]


def _essay_priority_pool(step_n, tier_data, version='V1'):
    """Returns priority-ordered list of ESSAY objects for L4M distractors."""
    sequence = tier_data['sequence']
    pool     = tier_data['pool']
    step     = sequence[step_n]

    used_ids = set()
    ordered  = []

    if step.get('model'):
        used_ids.add(step['model']['essay_id'])

    if version == 'V1':
        ch = step.get('challenge')
        if ch and ch['essay_id'] not in used_ids:
            ordered.append(ch); used_ids.add(ch['essay_id'])
        if step_n + 1 < len(sequence):
            for key in ['model', 'challenge']:
                e = sequence[step_n + 1].get(key)
                if e and e['essay_id'] not in used_ids:
                    ordered.append(e); used_ids.add(e['essay_id'])
    else:
        if step_n + 1 < len(sequence):
            for key in ['model', 'challenge']:
                e = sequence[step_n + 1].get(key)
                if e and e['essay_id'] not in used_ids:
                    ordered.append(e); used_ids.add(e['essay_id'])
        ch = step.get('challenge')
        if ch and ch['essay_id'] not in used_ids:
            ordered.append(ch); used_ids.add(ch['essay_id'])

    for essay in pool:
        if essay['essay_id'] not in used_ids:
            ordered.append(essay); used_ids.add(essay['essay_id'])

    return ordered


def _direction_argument_distractors(direction_tag, pools, n=4):
    """
    Finds n direction.argument texts from pool sentences where direction_tag differs.
    v5: uses dir_argument() helper since DIRECTION_LOOKUP is now nested.
    """
    dist_texts = []
    seen_tags  = {direction_tag}
    for pool in pools:
        for s in pool:
            dtag = s.get('direction_tag')
            if dtag and dtag not in seen_tags:
                dtext = dir_argument(dtag)
                if dtext and dtext not in dist_texts:
                    dist_texts.append(dtext)
                    seen_tags.add(dtag)
        if len(dist_texts) >= n:
            break
    return dist_texts[:n]


def _sentence_distractors_by_direction(target_direction_tag, pools, n=4, same_direction=False, exclude_ids=None):
    """
    Returns n distractor sentence canonical_texts from pools.
    same_direction=False: sentences with DIFFERENT non-null direction_tag (for P1/P2/P3)
    same_direction=True:  sentences with SAME direction_tag as target (for P4)
    Null-direction sentences are never eligible.
    """
    exclude_ids = exclude_ids or set()
    candidates = []
    seen_ids = set(exclude_ids)
    for pool in pools:
        for s in pool:
            sid = s.get('sentence_id', '')
            dtag = s.get('direction_tag')
            if sid in seen_ids or not dtag:
                continue
            if same_direction and dtag == target_direction_tag:
                candidates.append(s['canonical_text'])
                seen_ids.add(sid)
            elif not same_direction and dtag != target_direction_tag:
                candidates.append(s['canonical_text'])
                seen_ids.add(sid)
            if len(candidates) >= n:
                return candidates[:n]
    return candidates[:n]


def _paragraph_distractors(target_para_type, model, step_n, tier_data, n=4, version='V1'):
    """
    Returns n distractor paragraph texts (full paragraph, all sentences joined).
    Used by L3M_POV-P2. Same-essay body paragraphs first (different dominant direction),
    then tier pool body paragraphs as fallback.
    """
    target_dom = None
    candidates = []

    # Same-essay body paragraphs first
    for para in _BODY_PARAS:
        if para == target_para_type:
            continue
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue
        para_text = ' '.join(s['canonical_text'] for s in para_sents)
        candidates.append(para_text)

    # Tier pool fallback — always needed (max 3 body paras in essay)
    if len(candidates) < n:
        pools = build_distractor_pools(step_n, tier_data)
        if version == 'V2':
            pools = _pools_v2(pools)
        seen_texts = set(candidates)
        for pool in pools:
            # Group pool sentences by essay_id + paragraph_type
            essay_paras = {}
            for s in pool:
                if s['paragraph_type'] not in _BODY_PARAS:
                    continue
                key = (s.get('essay_id', ''), s['paragraph_type'])
                essay_paras.setdefault(key, []).append(s)

            for key, sents in essay_paras.items():
                para_text = ' '.join(s['canonical_text'] for s in sorted(sents, key=lambda s: s['order']))
                if para_text not in seen_texts:
                    candidates.append(para_text)
                    seen_texts.add(para_text)
                if len(candidates) >= n:
                    break
            if len(candidates) >= n:
                break

    return candidates[:n]


# ═══════════════════════════════════════════════════════════════════════════════
# L3M_POV — Paragraph MCQ (PoV Encoding) — NEW in v5, replaces old L3M
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L3M_POV(step_n, tier_data):
    """
    Two question types:
    P1: "Which PoV argument is this paragraph based on?" (one per body paragraph)
    P2: "Which paragraph is developing this PoV?" (one per unique direction_tag)
    """
    step  = tier_data['sequence'][step_n]
    model = step['model']
    pools = build_distractor_pools(step_n, tier_data)

    questions_v1 = []
    questions_v2 = []

    # ── P1: which PoV does this paragraph develop? ───────────────────────────
    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue

        dom_tag = dominant_direction(para_sents)
        if not dom_tag:
            continue

        full_text = ' '.join(s['canonical_text'] for s in para_sents)
        correct_arg = dir_argument(dom_tag)
        q_id = f"L3M_POV-P1-{para}"

        v1_dist = _direction_argument_distractors(dom_tag, pools)
        v2_dist = _direction_argument_distractors(dom_tag, _pools_v2(pools))

        v1_opts = shuffle_options(correct_arg, v1_dist)
        v2_opts = shuffle_options(correct_arg, v2_dist)

        base = {"id": q_id, "prompt": "Which PoV argument is this paragraph based on?", "context": full_text}
        questions_v1.append({**base, **v1_opts})
        questions_v2.append({**base, **v2_opts})

    # ── P2: which paragraph develops this PoV? ───────────────────────────────
    seen_directions = []
    para_by_dom = {}  # direction_tag -> paragraph_type

    for para in _BODY_PARAS:
        para_sents = sorted(
            [s for s in model['sentences'] if s['paragraph_type'] == para],
            key=lambda s: s['order']
        )
        if not para_sents:
            continue
        dom_tag = dominant_direction(para_sents)
        if dom_tag and dom_tag not in seen_directions:
            seen_directions.append(dom_tag)
            para_by_dom[dom_tag] = (para, para_sents)

    for dtag in seen_directions:
        para, para_sents = para_by_dom[dtag]
        correct_text = ' '.join(s['canonical_text'] for s in para_sents)
        context = dir_argument(dtag) + "\n\n" + dir_logic(dtag)
        q_id = f"L3M_POV-P2-{dtag}"

        v1_dist = _paragraph_distractors(para, model, step_n, tier_data, n=4, version='V1')
        v2_dist = _paragraph_distractors(para, model, step_n, tier_data, n=4, version='V2')

        v1_opts = shuffle_options(correct_text, v1_dist)
        v2_opts = shuffle_options(correct_text, v2_dist)

        base = {"id": q_id, "prompt": "Which paragraph is developing this PoV?", "context": context}
        questions_v1.append({**base, **v1_opts})
        questions_v2.append({**base, **v2_opts})

    return {
        "practice_code": "L3M_POV",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2M_POV — Sentence MCQ (PoV Encoding) — NEW in v5
# ═══════════════════════════════════════════════════════════════════════════════

# Rhetoric tags valid for each prompt type
_L2M_POV_P1_TAGS = {'example_1', 'example_2', 'advantage_1', 'advantage_2'}
_L2M_POV_P2_TAGS = {'explanation', 'cause_1', 'cause_2', 'effect_1', 'effect_2'}
_L2M_POV_P3_TAGS = {'problem_1', 'problem_2', 'disadvantage_1', 'disadvantage_2'}

_L2M_POV_PROMPTS = [
    ('P1', "Which sentence supports this PoV with a concrete example?",    _L2M_POV_P1_TAGS, False),
    ('P2', "Which sentence develops the reasoning behind this PoV?",       _L2M_POV_P2_TAGS, False),
    ('P3', "Which sentence identifies a problem or downside related to this PoV?", _L2M_POV_P3_TAGS, False),
    ('P4', "Which sentence is arguing for a different point of view?",     None,              True),
]

def assemble_L2M_POV(step_n, tier_data):
    """
    Per direction_tag in essay: up to 2 questions, priority P1→P2→P3→P4.
    Context = direction.argument + logic. Options = sentence canonical_texts.
    """
    step  = tier_data['sequence'][step_n]
    model = step['model']
    pools = build_distractor_pools(step_n, tier_data)

    # Find unique direction_tags in essay order
    direction_order = []
    for s in sorted(model['sentences'], key=lambda s: (_PARA_ORDER.index(s['paragraph_type']) if s['paragraph_type'] in _PARA_ORDER else 99, s['order'])):
        dtag = s.get('direction_tag')
        if dtag and dtag not in direction_order:
            direction_order.append(dtag)

    # Group sentences by direction_tag
    sents_by_dir = {}
    for s in model['sentences']:
        dtag = s.get('direction_tag')
        if dtag:
            sents_by_dir.setdefault(dtag, []).append(s)

    questions_v1 = []
    questions_v2 = []

    for dtag in direction_order:
        dir_sents = sents_by_dir.get(dtag, [])
        context = dir_argument(dtag) + "\n\n" + dir_logic(dtag)
        q_count = 0

        for prompt_code, prompt_text, valid_tags, is_reverse in _L2M_POV_PROMPTS:
            if q_count >= 2:
                break

            if is_reverse:
                # P4: find a sentence with a DIFFERENT non-null direction_tag
                other_sents = [s for s in model['sentences'] if s.get('direction_tag') and s['direction_tag'] != dtag]
                if not other_sents:
                    continue
                correct_sent = other_sents[0]
                correct_text = correct_sent['canonical_text']
                # Distractors: sentences with SAME direction_tag as target
                v1_dist = _sentence_distractors_by_direction(dtag, pools, n=4, same_direction=True, exclude_ids={correct_sent['sentence_id']})
                v2_dist = _sentence_distractors_by_direction(dtag, _pools_v2(pools), n=4, same_direction=True, exclude_ids={correct_sent['sentence_id']})
            else:
                # P1/P2/P3: find a sentence with this direction_tag AND matching rhetoric_tag
                matching = [s for s in dir_sents if s.get('rhetoric_tag') in valid_tags]
                if not matching:
                    continue
                correct_sent = matching[0]
                correct_text = correct_sent['canonical_text']
                # Distractors: sentences with DIFFERENT non-null direction_tag
                v1_dist = _sentence_distractors_by_direction(dtag, pools, n=4, same_direction=False, exclude_ids={correct_sent['sentence_id']})
                v2_dist = _sentence_distractors_by_direction(dtag, _pools_v2(pools), n=4, same_direction=False, exclude_ids={correct_sent['sentence_id']})

            q_id = f"L2M_POV-{prompt_code}-{dtag}"

            v1_opts = shuffle_options(correct_text, v1_dist)
            v2_opts = shuffle_options(correct_text, v2_dist)

            base = {"id": q_id, "prompt": prompt_text, "context": context}
            questions_v1.append({**base, **v1_opts})
            questions_v2.append({**base, **v2_opts})
            q_count += 1

    return {
        "practice_code": "L2M_POV",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L1M — Lexical MCQ (v5: body sentences only, first noun-phrase per sentence)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L1M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']

    # Build same-essay noun-phrase pool for distractors
    all_noun_phrases = []
    for s in model['sentences']:
        for li in s.get('lexical_items', []):
            p = pos_tag(li['phrase'])
            if p == 'NOUN' and li['phrase'] not in all_noun_phrases:
                all_noun_phrases.append(li['phrase'])

    def pick_phrase_distractors(phrase, n=4):
        seen = {phrase}; result = []
        for p in all_noun_phrases:
            if p not in seen:
                result.append(p); seen.add(p)
            if len(result) >= n:
                break
        return result[:n]

    body_sents = _body_sentences(model)
    questions_v1 = []
    questions_v2 = []

    for s in body_sents:
        # Find first noun-phrase lexical item
        noun_li = None
        for li in s.get('lexical_items', []):
            if pos_tag(li['phrase']) == 'NOUN':
                noun_li = li
                break
        if not noun_li:
            continue

        phrase = noun_li['phrase']
        actual = find_phrase_in_text(phrase, s['canonical_text'])
        blank_context = s['canonical_text'].replace(actual, '___', 1) if actual else s['canonical_text']
        distractors = pick_phrase_distractors(phrase)

        q_id = f"L1M-{s['paragraph_type']}-S{s['order']}"
        base = {
            "id": q_id,
            "prompt": "All options appear in the essay — which phrase belongs here?",
            "context": blank_context,
        }
        questions_v1.append({**base, **shuffle_options(phrase, distractors)})
        questions_v2.append({**base, **shuffle_options(phrase, distractors)})

    return {
        "practice_code": "L1M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L2M — Sentence MCQ (v5: rhetoric-only, body sentences, cap 6)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L2M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']

    # Get rhetoric_tags that appear in this essay's structure_type
    structure_tags = list(dict.fromkeys(s['rhetoric_tag'] for s in model['sentences']))

    body_sents = _body_sentences(model)
    questions_v1 = []
    questions_v2 = []

    for s in body_sents[:6]:  # v5: cap at 6
        correct_label = RHETORIC_LABEL_LOOKUP.get(s['rhetoric_tag'], s['rhetoric_tag'])
        other_labels = [RHETORIC_LABEL_LOOKUP.get(t, t) for t in structure_tags if t != s['rhetoric_tag']]

        q_id = f"L2M-{s['paragraph_type']}-S{s['order']}"
        base = {
            "id": q_id,
            "prompt": "What structural role does this sentence play?",
            "context": s['canonical_text'],
        }
        questions_v1.append({**base, **shuffle_options(correct_label, other_labels[:4])})
        questions_v2.append({**base, **shuffle_options(correct_label, other_labels[:4])})

    return {
        "practice_code": "L2M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


# ═══════════════════════════════════════════════════════════════════════════════
# L4M — Essay MCQ (v5: 2 questions only, structure question removed)
# ═══════════════════════════════════════════════════════════════════════════════
def assemble_L4M(step_n, tier_data):
    step  = tier_data['sequence'][step_n]
    model = step['model']

    correct_q = model['question']

    # Build correct PoV pair string
    model_dir_args = []
    seen_dtags     = set()
    for s in model['sentences']:
        dtag = s.get('direction_tag')
        if dtag and dtag not in seen_dtags:
            arg = dir_argument(dtag)
            if arg: model_dir_args.append(arg)
            seen_dtags.add(dtag)
    correct_pov = ' / '.join(model_dir_args) if model_dir_args else '—'

    version_data = {}
    for version in ['V1', 'V2']:
        essay_pool = _essay_priority_pool(step_n, tier_data, version)

        # L4M-1 distractors: questions from pool
        dist_qs = []; seen_qs = {correct_q}
        for essay in essay_pool:
            q = essay.get('question', '')
            if q and q not in seen_qs: dist_qs.append(q); seen_qs.add(q)
            if len(dist_qs) >= 4: break

        # L4M-2 distractors: direction pairs from pool
        dist_povs = []; seen_povs = {correct_pov}
        for essay in essay_pool:
            args = []; seen_tags = set()
            for s in essay.get('sentences', []):
                dtag = s.get('direction_tag')
                if dtag and dtag not in seen_tags:
                    arg = dir_argument(dtag)
                    if arg: args.append(arg)
                    seen_tags.add(dtag)
            pov_str = ' / '.join(args) if args else None
            if pov_str and pov_str not in seen_povs: dist_povs.append(pov_str); seen_povs.add(pov_str)
            if len(dist_povs) >= 4: break

        version_data[version] = {
            'q1': shuffle_options(correct_q, dist_qs),
            'q2': shuffle_options(correct_pov, dist_povs),
        }

    # v5: only 2 questions (structure question L4M-3 removed). context="" for essay-level.
    questions_v1 = [
        {"id": "L4M-1", "prompt": "Which IELTS question does this essay respond to?", "context": "", **version_data['V1']['q1']},
        {"id": "L4M-2", "prompt": "Which pair of PoVs does this essay develop?", "context": "", **version_data['V1']['q2']},
    ]
    questions_v2 = [
        {"id": "L4M-1", "prompt": "Which IELTS question does this essay respond to?", "context": "", **version_data['V2']['q1']},
        {"id": "L4M-2", "prompt": "Which pair of PoVs does this essay develop?", "context": "", **version_data['V2']['q2']},
    ]

    return {
        "practice_code": "L4M",
        "versions": [
            {"version": "V1", "questions": questions_v1},
            {"version": "V2", "questions": questions_v2},
        ]
    }


print('✅ MCQ assemblers ready: L3M_POV, L2M_POV, L1M, L2M, L4M')

---
## Cell 9 — Assemble `sentences` JSONB helper

Strips internal fields from essay sentences. The `sentences` JSONB on `prep_unit`
is for display only (peek modal, L4W reference panels). No `similar_phrases` or
`hint_sentences` — those live inside `practices`.

In [9]:
def build_sentences_jsonb(model_essay):
    """
    Build the sentences JSONB for a prep_unit row.
    Strips hint_sentences, similar_phrases — those are consumed by practices.
    Adds rhetoric_label and pos on lexical items.
    """
    sentences = []
    for s in model_essay['sentences']:
        sentences.append({
            'sentence_id':    s['sentence_id'],
            'paragraph_type': s['paragraph_type'],
            'order':          s['order'],
            'canonical_text': s['canonical_text'],
            'rhetoric_tag':   s['rhetoric_tag'],
            'direction_tag':  s.get('direction_tag'),
            'rhetoric_label': RHETORIC_LABEL_LOOKUP.get(s.get('rhetoric_tag', ''), s.get('rhetoric_tag', '')),
            'lexical_items':  [
                {'phrase': li['phrase'], 'pos': pos_tag(li['phrase'])}
                for li in s.get('lexical_items', [])
            ],
            'syntax_items':   s.get('syntax_items', []),
        })
    return sentences


print('✅ build_sentences_jsonb ready')

✅ build_sentences_jsonb ready


---
## Cell 10 — Main assembly loop

Iterates over all tiers and steps, assembles prep_unit rows.
**Deduplication:** Each model essay gets ONE prep_unit row, even if it appears
in both tier_50 and tier_80. The tier assignment lives in `tier_unit_sequence`.

In [ ]:
# Track which unit_ids we've already assembled (deduplication)
assembled_units = {}  # unit_id -> prep_unit row

print('Assembling prep-units (v5: 13 practices)...')

for tier_name in ['tier_50', 'tier_80']:
    tier_data = snapshot[tier_name]
    n_steps   = len(tier_data['sequence'])
    print(f'  {tier_name}: {n_steps} steps')

    for step_n in range(n_steps):
        step  = tier_data['sequence'][step_n]
        model = step['model']
        unit_id = model['essay_id']

        # Skip if already assembled (shared unit between tiers)
        if unit_id in assembled_units:
            continue

        # Build the 13-practice ordered array (v5 — L4S dropped)
        practices = [
            assemble_P0(step),                            # 0  P0
            assemble_POV_INTRO(step),                     # 1  POV_INTRO
            assemble_L3M_POV(step_n, tier_data),          # 2  L3M_POV
            assemble_L2M_POV(step_n, tier_data),          # 3  L2M_POV
            assemble_L4M(step_n, tier_data),              # 4  L4M
            assemble_L2M(step_n, tier_data),              # 5  L2M
            assemble_L1M(step_n, tier_data),              # 6  L1M
            assemble_L1S(step),                           # 7  L1S
            assemble_L2S(step),                           # 8  L2S
            assemble_L3S(step),                           # 9  L3S
            assemble_L1F(step),                           # 10 L1F
            assemble_L2F(step),                           # 11 L2F
            assemble_L4W(step),                           # 12 L4W
        ]

        # Build sentences JSONB
        sentences_jsonb = build_sentences_jsonb(model)

        assembled_units[unit_id] = {
            'unit_id':        unit_id,
            'batch_id':       BATCH_ID,
            'question':       model['question'],
            'structure_type': model['structure_type'],
            'sentences':      sentences_jsonb,
            'practices':      practices,
        }

print(f'\nTotal unique prep-units: {len(assembled_units)}')
print(f'Practice codes per unit: {[p["practice_code"] for p in list(assembled_units.values())[0]["practices"]]}')
print('✅ Assembly complete (v5: 13 practices per unit — L4S dropped)')

---
## Cell 11 — Write `prep_unit` to Supabase

```sql
-- Reference: create this table in Supabase dashboard first
CREATE TABLE prep_unit (
    unit_id        UUID PRIMARY KEY,
    batch_id       UUID NOT NULL,
    question       TEXT NOT NULL,
    structure_type TEXT NOT NULL,
    sentences      JSONB NOT NULL,
    practices      JSONB NOT NULL
);
```

In [11]:
import json

# Build rows — JSONB columns need to be JSON strings for Supabase
prep_unit_rows = []
for unit_id, unit in assembled_units.items():
    prep_unit_rows.append({
        'unit_id':        unit['unit_id'],
        'batch_id':       unit['batch_id'],
        'question':       unit['question'],
        'structure_type': unit['structure_type'],
        'sentences':      json.dumps(unit['sentences']),
        'practices':      json.dumps(unit['practices']),
    })

print(f'Rows to insert: {len(prep_unit_rows)}')

# Insert in batches of 10 (JSONB rows can be large)
batch_size = 10
for i in range(0, len(prep_unit_rows), batch_size):
    batch = prep_unit_rows[i:i + batch_size]
    resp = supabase.table('prep_unit').upsert(batch).execute()
    print(f'  Batch {i//batch_size + 1}: wrote {len(resp.data)} rows')

print(f'\n✅ Wrote {len(prep_unit_rows)} rows to prep_unit')

Rows to insert: 14
  Batch 1: wrote 10 rows
  Batch 2: wrote 4 rows

✅ Wrote 14 rows to prep_unit


---
## Cell 12 — Save data.json to Google Drive (backup)

Saves the complete output as a JSON file for inspection and backup.
The primary output is already in Supabase — this is just a readable copy.

In [12]:
import json, os

# Build the complete data.json shape
data_json = {
    "batch_id": BATCH_ID,
    "prep_units": list(assembled_units.values()),
}

OUTPUT_FILE = 'data.json'
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(data_json, f, indent=2, ensure_ascii=False)

size_mb = os.path.getsize(OUTPUT_FILE) / 1024 / 1024
print(f'✅ Saved {OUTPUT_FILE} ({size_mb:.2f} MB)')
print(f'   prep_units: {len(data_json["prep_units"])}')

# Upload to Google Drive
from googleapiclient.http import MediaFileUpload

UPLOAD_NAME = 'data.json'
existing = service.files().list(
    q=f"name='{UPLOAD_NAME}' and '{FOLDER_ID}' in parents and trashed=false",
    fields='files(id)'
).execute().get('files', [])

media = MediaFileUpload(OUTPUT_FILE, mimetype='application/json', resumable=False)
if existing:
    service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    print(f'♻️  Replaced existing {UPLOAD_NAME} in Drive')
else:
    meta = {'name': UPLOAD_NAME, 'parents': [FOLDER_ID]}
    service.files().create(body=meta, media_body=media, fields='id').execute()
    print(f'✅ Uploaded {UPLOAD_NAME} to Drive')

✅ Saved data.json (3.87 MB)
   prep_units: 14
✅ Uploaded data.json to Drive


---
## Cell 13 — Verification checklist

Validates prep_unit rows in Supabase and cross-checks with P2 output.

In [ ]:
passes   = []
failures = []

def check(label, condition, detail=''):
    if condition:
        passes.append(f'  ✅  {label}')
    else:
        failures.append(f'  ❌  {label}' + (f'\n       → {detail}' if detail else ''))

# v5: 13 practices in this exact order (L4S dropped)
EXPECTED_PRACTICE_CODES = ['P0', 'POV_INTRO', 'L3M_POV', 'L2M_POV', 'L4M', 'L2M', 'L1M', 'L1S', 'L2S', 'L3S', 'L1F', 'L2F', 'L4W']
VERSIONED_CODES   = {'L3M_POV', 'L2M_POV', 'L4M', 'L2M', 'L1M', 'L1S', 'L2S', 'L3S', 'L1F'}
UNVERSIONED_CODES = {'P0', 'POV_INTRO', 'L2F', 'L4W'}

# ── Check Supabase rows ───────────────────────────────────────────────────
pu_check = supabase.table('prep_unit').select('unit_id, batch_id, practices').eq('batch_id', BATCH_ID).execute()
check('prep_unit has rows', len(pu_check.data) > 0, f'Got {len(pu_check.data)}')
check('prep_unit count matches', len(pu_check.data) == len(assembled_units), f'Expected {len(assembled_units)}, got {len(pu_check.data)}')

# ── Check practice structure ───────────────────────────────────────────────
for row in pu_check.data[:5]:  # spot-check first 5
    practices = json.loads(row['practices']) if isinstance(row['practices'], str) else row['practices']
    uid = row['unit_id'][:8]

    check(f'{uid}… has 13 practices', len(practices) == 13, f'Got {len(practices)}')

    codes = [p.get('practice_code') for p in practices]
    check(f'{uid}… practice codes match v5 spec', codes == EXPECTED_PRACTICE_CODES, f'Got {codes}')

    for p in practices:
        code_name = p.get('practice_code', '?')
        if code_name in VERSIONED_CODES:
            check(f'{uid}… {code_name} has versions[]', 'versions' in p, f'Missing versions on {code_name}')
            if 'versions' in p:
                v_labels = [v.get('version') for v in p['versions']]
                check(f'{uid}… {code_name} versions are V1,V2', v_labels == ['V1', 'V2'], f'Got {v_labels}')
                # Check all MCQ questions have context field
                if code_name in {'L3M_POV', 'L2M_POV', 'L4M', 'L2M', 'L1M'}:
                    for v in p['versions']:
                        for q in v.get('questions', []):
                            check(f'{uid}… {code_name} Q {q.get("id","")} has context', 'context' in q, f'Missing context')
        elif code_name in UNVERSIONED_CODES:
            check(f'{uid}… {code_name} has NO versions', 'versions' not in p, f'{code_name} should not have versions')

    # ── v5-specific checks ──────────────────────────────────────────────────
    # POV_INTRO must have directions[]
    pov_intro = practices[1]
    check(f'{uid}… POV_INTRO has directions[]', 'directions' in pov_intro and len(pov_intro.get('directions', [])) > 0, f'Got {pov_intro.keys()}')
    for d in pov_intro.get('directions', []):
        check(f'{uid}… POV_INTRO dir has argument+logic', 'argument' in d and 'logic' in d, f'Got keys: {d.keys()}')

    # L4M should have exactly 2 questions (not 3)
    l4m = practices[4]
    if 'versions' in l4m:
        n_q = len(l4m['versions'][0].get('questions', []))
        check(f'{uid}… L4M has 2 questions', n_q == 2, f'Got {n_q}')

    # L2M should have cap 6 and body sentences only
    l2m = practices[5]
    if 'versions' in l2m:
        n_q = len(l2m['versions'][0].get('questions', []))
        check(f'{uid}… L2M has ≤6 questions', n_q <= 6, f'Got {n_q}')

    # L1S should have cap 10
    l1s = practices[7]
    if 'versions' in l1s:
        n_q = len(l1s['versions'][0].get('questions', []))
        check(f'{uid}… L1S has ≤10 questions', n_q <= 10, f'Got {n_q}')

    # L1F body only, cap 10
    l1f = practices[10]
    if 'versions' in l1f:
        n_q = len(l1f['versions'][0].get('questions', []))
        check(f'{uid}… L1F has ≤10 questions', n_q <= 10, f'Got {n_q}')

    # L2F body only, cap 5
    l2f = practices[11]
    n_q = len(l2f.get('questions', []))
    check(f'{uid}… L2F has ≤5 questions', n_q <= 5, f'Got {n_q}')

# ── Cross-check with tier_unit_sequence ────────────────────────────────────
tus_data = supabase.table('tier_unit_sequence').select('unit_id').eq('batch_id', BATCH_ID).execute()
tus_unit_ids = {r['unit_id'] for r in tus_data.data}
pu_unit_ids  = {r['unit_id'] for r in pu_check.data}
missing = tus_unit_ids - pu_unit_ids
check('All tier_unit_sequence unit_ids have prep_unit rows', len(missing) == 0, f'Missing: {list(missing)[:3]}')

# ── Print results ──────────────────────────────────────────────────────────
print('=' * 60)
print('PIPELINE 3 v5 — VERIFICATION')
print('=' * 60)
for msg in passes:
    print(msg)
for msg in failures:
    print(msg)
print('=' * 60)
print(f'  {len(passes)} passed   {len(failures)} failed')

if not failures:
    print('\n🎉 Pipeline 3 v5 complete — all checks passed.')
    print(f'   batch_id: {BATCH_ID}')
    print(f'   prep_units: {len(assembled_units)}')
    print(f'   practices per unit: 13')
    print(f'   practice_codes: {EXPECTED_PRACTICE_CODES}')  # 13 codes
    print('   The School can now ingest this data.')
else:
    print('\n🚨 Fix failures above before proceeding.')